In [1]:
1+1

2

In [2]:
import os
import sys
import re
from pathlib import Path

import pandas as pd
import numpy as np

from feature_engine_parts.fe_parts_V2.preprocessors.payment_pattern_aggregator import PaymentPatternsAggregatorV2
from feature_engine_parts.fe_parts_V2.preprocessors.date_diff import DateDiffV2
from model_engine.feature_engine_V2.listed_objects_engines import MapperV2

# Truist eval-workbook writer / formatter
sys.path.insert(0, '/home/jag/client-project-truist/autoplccv2/playground/jag/modeling/eval')
import functions as F

from configs import EQUIFAX, EXPERIAN, TRANSUNION, DATA_DIR, unmapped_dir, prepped_dir
from helpers import (
    MISSING_DATA_CHARS,
    PROJECT_COLS,
    MONTH_RANGES,
    load_asset_json,
    get_aggregator_params,
    get_relevant_mapping_steps,
)

SPLIT = 'train'

# Inputs and outputs both live under DATA_DIR (config-driven, not hardcoded).
# Inputs:  payment_processing_research_data/<bureau>/<SPLIT>/unmapped/
# Outputs: payment_processing_research_data/analysis/denominator_analysis_<SPLIT>{,_formatted}.xlsx
ANALYSIS_DIR = Path(DATA_DIR) / 'analysis'
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
OUT_XLSX     = ANALYSIS_DIR / f'denominator_analysis_{SPLIT}.xlsx'
OUT_XLSX_FMT = OUT_XLSX.with_name(OUT_XLSX.stem + '_formatted.xlsx')

print('DATA_DIR     =', DATA_DIR)
print('ANALYSIS_DIR =', ANALYSIS_DIR)
print('OUT_XLSX     =', OUT_XLSX)

BUREAU_CFGS = {'equifax': EQUIFAX, 'experian': EXPERIAN, 'transunion': TRANSUNION}

You have an incompatible version of 'pyarrow' installed (11.0.0), please install a version that adheres to: 'pyarrow>=14.0.1; extra == "pandas"'


DATA_DIR     = /home/jag/payment-processor-research/payment_processing_research_data
ANALYSIS_DIR = /home/jag/payment-processor-research/payment_processing_research_data/analysis
OUT_XLSX     = /home/jag/payment-processor-research/payment_processing_research_data/analysis/denominator_analysis_train.xlsx


In [3]:
import inspect

print(inspect.getsource(PaymentPatternsAggregatorV2._construct_trended_features))



    def _construct_trended_features(self, data, new_ppt):
        for month_range in self.month_ranges:
            trimmed = new_ppt.str[:month_range]
            effective_month_count = self._get_effective_month_range(trimmed, month_range)
            for rate, values in self._rate.items():
                count = self._get_count(trimmed, values)
                data[f"number_{rate}_{month_range}_months{self.name}"] = count.astype(self.PANDAS_DTYPES["numeric"])
                data[f"percent_{rate}_{month_range}_months{self.name}"] = (count / effective_month_count).astype(
                    self.PANDAS_DTYPES["numeric"]
                )
        return data



In [4]:
# Step 1 -- discover the per-(bureau, SPLIT) unmapped chunk files. We DO NOT
# load any data here -- step 2 streams them one part file at a time.
import gc

unmapped_parts = {}
for bureau, cfg in BUREAU_CFGS.items():
    d = Path(unmapped_dir(cfg, SPLIT))
    parts = sorted(d.glob('part-*.parquet')) if d.exists() else []
    if not parts:
        print(f'[{bureau}]   MISSING {d} -- run save_unmapped_data.ipynb first')
        continue
    unmapped_parts[bureau] = parts
    total_size_mb = sum(p.stat().st_size for p in parts) / (1024 * 1024)
    print(f'[{bureau}]   {len(parts)} chunks, {total_size_mb:.1f} MB on disk')

[equifax]   582 chunks, 1373.7 MB on disk
[experian]   443 chunks, 2203.3 MB on disk
[transunion]   534 chunks, 1274.3 MB on disk


In [5]:
# Step 2 -- CHUNKED prep. Mirrors `map_and_save_mapped_data.ipynb`:
# for each (bureau, chunk) we read one unmapped part file -> run
# prep_bureau on it -> save the result to
#   payment_processing_research_data/<bureau>/<SPLIT>/prepped/part-NNNNN.parquet
# Memory stays bounded to ~one chunk during the expensive mapping step.
# After all chunks are prepped, we re-read the prepped dir into prepped[bureau]
# so the existing analysis cells below work unchanged.
#
# prep_bureau(df_chunk) does, per chunk:
#   filter mapping steps -> project input to ONLY raw cols those steps need
#   -> slim MapperV2 -> project to PROJECT_COLS -> DateDiffV2
#   -> agg._construct_payment_pattern_cols

def prep_bureau(bureau, trade_df):
    """Returns to_use_for_payment_processing with `zest_payment_pattern` populated."""
    asset = load_asset_json(bureau)

    # 1. STEP slim: only converters producing columns we use downstream.
    needed_outputs = list(PROJECT_COLS[bureau])
    steps = get_relevant_mapping_steps(asset, needed_outputs)

    # 2. COLUMN slim: only the raw input columns those steps actually read.
    raw_inputs = sorted({s['params']['raw_feature'] for s in steps})

    # Project the unmapped df down to only the raw columns the kept steps read.
    trade_df_subset = trade_df[raw_inputs].copy()

    # Run the slimmed MapperV2 on the slimmed df.
    mapper = MapperV2(api=steps)
    trade_df_mapped = mapper.transform(trade_df_subset)

    # Final projection to PROJECT_COLS for the aggregator.
    cols = [c for c in PROJECT_COLS[bureau] if c in trade_df_mapped.columns]
    to_use_for_payment_processing = trade_df_mapped[cols].copy()

    # DateDiffV2: (date_of_request - rptDate) / 30.436875 days -> months_since_rptDate
    date_diff = DateDiffV2(feature='rptDate', reference_feature='date_of_request',
                           new_feature='months_since_rptDate')
    to_use_for_payment_processing = date_diff.transform(to_use_for_payment_processing)

    # Combine into zest_payment_pattern via the aggregator helper.
    agg = PaymentPatternsAggregatorV2(**get_aggregator_params(asset))
    to_use_for_payment_processing['zest_payment_pattern'] = agg._construct_payment_pattern_cols(
        to_use_for_payment_processing
    )
    return to_use_for_payment_processing


# STREAMING loop: read one chunk -> prep -> save to prepped/ -> free memory.
# Each input `part-NNNNN.parquet` maps 1:1 to an output `part-NNNNN.parquet`.
prepped = {}
for bureau, parts in unmapped_parts.items():
    cfg     = BUREAU_CFGS[bureau]
    out_dir = Path(prepped_dir(cfg, SPLIT))
    out_dir.mkdir(parents=True, exist_ok=True)
    for old in out_dir.glob('part-*.parquet'):
        old.unlink()

    print(f'\n[{bureau}]  streaming {len(parts)} chunks -> {out_dir}')
    total_rows = 0
    for i, in_path in enumerate(parts):
        chunk        = pd.read_parquet(in_path)
        prepped_chunk = prep_bureau(bureau, chunk)
        prepped_chunk.to_parquet(out_dir / f'part-{i:05d}.parquet', index=False)
        total_rows += len(prepped_chunk)

        if (i + 1) % 50 == 0 or i == len(parts) - 1:
            print(f'[{bureau}]   {i + 1}/{len(parts)} chunks prepped ({total_rows:,} rows)')

        del chunk, prepped_chunk
        gc.collect()

    # Re-read the prepped chunks for the downstream analysis cells. (This
    # loads the full prepped data into memory, which is ~5 small cols instead
    # of the ~28 raw cols in the unmapped dir.)
    prepped[bureau] = pd.read_parquet(out_dir)
    sample = prepped[bureau]['zest_payment_pattern'].dropna()
    print(f'[{bureau}]   done -- reread {len(prepped[bureau]):,} rows; '
          f'example pattern: {sample.iloc[0][:60] if len(sample) else "(none)"} ...')


[equifax]  streaming 582 chunks -> /home/jag/payment-processor-research/payment_processing_research_data/equifax/train/prepped
[equifax]   50/582 chunks prepped (5,000,000 rows)
[equifax]   100/582 chunks prepped (10,000,000 rows)
[equifax]   150/582 chunks prepped (15,000,000 rows)
[equifax]   200/582 chunks prepped (20,000,000 rows)
[equifax]   250/582 chunks prepped (25,000,000 rows)
[equifax]   300/582 chunks prepped (30,000,000 rows)
[equifax]   350/582 chunks prepped (35,000,000 rows)
[equifax]   400/582 chunks prepped (40,000,000 rows)
[equifax]   450/582 chunks prepped (45,000,000 rows)
[equifax]   500/582 chunks prepped (50,000,000 rows)
[equifax]   550/582 chunks prepped (55,000,000 rows)
[equifax]   582/582 chunks prepped (58,130,132 rows)
[equifax]   done -- reread 58,130,132 rows; example pattern: ########################1*********************************** ...

[experian]  streaming 443 chunks -> /home/jag/payment-processor-research/payment_processing_research_data/exper

In [ ]:
# OPTIONAL fast-path -- re-load `prepped` from the on-disk parquet chunks
# WITHOUT re-running the chunked prep loop above. Useful after a kernel
# restart: the prep cell may have already written every part-*.parquet to
# `<bureau>/<SPLIT>/prepped/`, so there's no need to redo the mapping +
# aggregator work. No-op for any bureau already in memory.
if 'prepped' not in globals() or not prepped:
    prepped = {}

for bureau, cfg in BUREAU_CFGS.items():
    if bureau in prepped and len(prepped[bureau]):
        print(f'[{bureau}]  already in memory ({len(prepped[bureau]):,} rows) -- skipped')
        continue
    d = Path(prepped_dir(cfg, SPLIT))
    if d.exists() and list(d.glob('part-*.parquet')):
        prepped[bureau] = pd.read_parquet(d)
        print(f'[{bureau}]  loaded {len(prepped[bureau]):,} rows from {d}')
    else:
        print(f'[{bureau}]  no prepped/ chunks on disk -- run the prep cell above first')

In [6]:
# Step 3 -- BY HAND. For each bureau and each month_range, compute old + new
# effective month range with pure pandas string ops. We do NOT call into the
# aggregator's _get_effective_month_range or _construct_trended_features --
# everything here is open code.
#
# OLD method (current upstream code in _construct_trended_features):
#     eff_old = month_range - count('#' in trimmed)
#
# NEW method (proposed fix):
#     eff_new = trimmed.str.len() - count('#' in trimmed) - count(missing_char in trimmed)
#
# The result for each bureau is a DataFrame with one column pair per month_range:
#     eff_old_3, eff_new_3, eff_old_6, eff_new_6, ..., eff_old_48, eff_new_48
# so the per-row math is inspectable after this cell.

eff = {}
for bureau, df in prepped.items():
    missing_char = MISSING_DATA_CHARS[bureau]
    print(f'\n[{bureau}]  missing_char={missing_char!r}  -- computing for {MONTH_RANGES}')

    out = df[['ZEST_KEY', 'zest_payment_pattern']].copy()
    ppt = df['zest_payment_pattern'].fillna('')

    for m in MONTH_RANGES:
        trimmed       = ppt.str[:m]
        hash_count    = trimmed.str.count('#')
        missing_count = trimmed.str.count(re.escape(missing_char))
        obs_len       = trimmed.str.len()

        out[f'trimmed_{m}']  = trimmed
        out[f'eff_old_{m}']  = m       - hash_count
        out[f'eff_new_{m}']  = obs_len - hash_count - missing_count

    eff[bureau] = out
    print(f'[{bureau}]  per-row eff_old/eff_new columns added for every month_range. '
          f'Shape: {out.shape}')

# Quick spot-check: first 3 rows of equifax to confirm the math looks right.
if 'equifax' in eff:
    show_cols = ['ZEST_KEY', 'trimmed_6', 'eff_old_6', 'eff_new_6',
                 'trimmed_24', 'eff_old_24', 'eff_new_24']
    eff['equifax'][show_cols].head(3)


[equifax]  missing_char='*'  -- computing for [3, 6, 12, 24, 48]
[equifax]  per-row eff_old/eff_new columns added for every month_range. Shape: (58130132, 17)

[experian]  missing_char='-'  -- computing for [3, 6, 12, 24, 48]
[experian]  per-row eff_old/eff_new columns added for every month_range. Shape: (44246273, 17)

[transunion]  missing_char='X'  -- computing for [3, 6, 12, 24, 48]
[transunion]  per-row eff_old/eff_new columns added for every month_range. Shape: (53313028, 17)


In [19]:
# Step 4 -- aggregate per-row eff_old / eff_new into the two summary tables.
# Tab layout:
#   - 'overall'   : Table 1 + Table 2, pooled across ALL bureaus
#   - 'equifax'   : Table 1 + Table 2, restricted to equifax rows
#   - 'experian'  : Table 1 + Table 2, restricted to experian rows
#   - 'transunion': Table 1 + Table 2, restricted to transunion rows
#
# Same two table shapes in every tab; the only thing that changes between
# tabs is which rows go in.

def table_all(eff_dict):
    """Table 1: one row per month_range, computed across the pool of
    eff_old/eff_new rows from every (bureau,row) in eff_dict.

    Pass a single-bureau dict for per-bureau tabs; pass the full dict for
    the 'overall' tab. Aggregates partial sums per bureau (no giant concat)
    so combining 3 bureaus stays cheap.
    """
    rows = []
    for m in MONTH_RANGES:
        n         = 0
        sum_old   = 0.0
        sum_new   = 0.0
        n_changed = 0
        for df in eff_dict.values():
            old = df[f'eff_old_{m}']
            new = df[f'eff_new_{m}']
            n         += len(df)
            sum_old   += float(old.sum())
            sum_new   += float(new.sum())
            n_changed += int((old != new).sum())
        rows.append({
            'month_range':        m,
            'n_rows':             n,
            'avg_eff_old':        round(sum_old / n, 2)         if n else float('nan'),
            'avg_eff_new':        round(sum_new / n, 2)         if n else float('nan'),
            'pct_rows_changed_%': round(n_changed / n * 100, 2) if n else float('nan'),
        })
    return pd.DataFrame(rows)


def table_only_changed(eff_dict):
    """Table 2: one row per month_range, computed across rows where
    eff_old != eff_new, pooled across every df in eff_dict. Adds 25/50/75
    percentiles for both methods.
    """
    rows = []
    for m in MONTH_RANGES:
        old_chunks, new_chunks = [], []
        for df in eff_dict.values():
            old = df[f'eff_old_{m}']
            new = df[f'eff_new_{m}']
            mask = old != new
            old_chunks.append(old[mask])
            new_chunks.append(new[mask])
        old_v = pd.concat(old_chunks, ignore_index=True) if old_chunks else pd.Series([], dtype=float)
        new_v = pd.concat(new_chunks, ignore_index=True) if new_chunks else pd.Series([], dtype=float)
        n = len(old_v)
        rows.append({
            'month_range':  m,
            'n_changed':    int(n),
            'avg_eff_old':  round(old_v.mean(), 2)          if n else float('nan'),
            'avg_eff_new':  round(new_v.mean(), 2)          if n else float('nan'),
            'avg_change':   round((new_v - old_v).mean(), 2) if n else float('nan'),
            'p25_eff_old':  round(old_v.quantile(0.25), 2)  if n else float('nan'),
            'p50_eff_old':  round(old_v.quantile(0.50), 2)  if n else float('nan'),
            'p75_eff_old':  round(old_v.quantile(0.75), 2)  if n else float('nan'),
            'p25_eff_new':  round(new_v.quantile(0.25), 2)  if n else float('nan'),
            'p50_eff_new':  round(new_v.quantile(0.50), 2)  if n else float('nan'),
            'p75_eff_new':  round(new_v.quantile(0.75), 2)  if n else float('nan'),
        })
    return pd.DataFrame(rows)


# Every tab carries the same two tables. The only thing that changes between
# tabs is which subset of `eff` is passed in -- the whole dict for 'overall',
# a single-bureau dict for the per-bureau tabs.
TABS = {}

if eff:
    TABS['overall'] = {
        'All tradelines':                                       table_all(eff),
        'Only tradelines whose effective month range changed':  table_only_changed(eff),
    }

for bureau, eff_df in eff.items():
    single = {bureau: eff_df}
    TABS[bureau] = {
        'All tradelines':                                       table_all(single),
        'Only tradelines whose effective month range changed':  table_only_changed(single),
    }

for tab, sections in TABS.items():
    print(f'{tab:11s}  ->  {sum(len(t) for t in sections.values()):3d} rows / {len(sections)} sections')

overall      ->   10 rows / 2 sections
equifax      ->   10 rows / 2 sections
experian     ->   10 rows / 2 sections
transunion   ->   10 rows / 2 sections


In [20]:
_GENERAL_NOTES = [
    'NOTES',
    '',
    'This workbook compares the OLD effective-month-range computation in PaymentPatternsAggregatorV2 against the NEW (proposed) one for each per-month_range trended feature.',
    '',
    'Pipeline:',
    '  - Step 1: load unmapped chunks from payment_processing_research_data/<bureau>/<SPLIT>/unmapped/.',
    '  - Step 2: combine raw bureau columns into zest_payment_pattern using the aggregator helper: MapperV2 -> project subset -> DateDiffV2 -> agg._construct_payment_pattern_cols.',
    '  - Step 3: BY HAND -- for each month_range in [3, 6, 12, 24, 48], count # and the bureau\'s missing-data char with pure pandas string ops. No call into _get_effective_month_range or _construct_trended_features.',
    '  - Step 4: aggregate to per-(bureau, month_range) summary tables.',
    '',
    'OLD method (current upstream code in _construct_trended_features):',
    "  - eff_old = month_range - count('#' in trimmed)",
    "  - Only '#' (pre-report-date filler) is subtracted. Bureau missing-data codes ('*', '-', 'X') are silently treated as observed paid-as-agreed months, inflating the denominator and biasing percent_<DQ>_<window> features toward zero.",
    '',
    'NEW method (proposed fix in the modified PaymentPatternsAggregatorV2):',
    "  - eff_new = trimmed.str.len() - count('#' in trimmed) - count(missing_char in trimmed)",
    "  - Subtracts the bureau-specific missing-data char in addition to '#', anywhere in the trimmed window (not just trailing).",
    '  - Anchors the denominator on the OBSERVED string length, not the nominal month_range.',
    '',
    'Tables in each tab:',
    '  - Table 1 (All tradelines): for each month_range -- avg_eff_old, avg_eff_new, pct_rows_changed.',
    '  - Table 2 (Only changed rows): filtered to eff_old != eff_new. avg_eff_old, avg_eff_new, avg_change, and 25/50/75 percentiles for both.',
    '',
    'Per-bureau missing-data char (sources: spec PDFs in old_context/):',
    "  - Equifax    '*' -- 'Rate/Status was not available for that month'   (System-to-System TotalView Programming Guide, p. 3-30)",
    "  - Experian   '-' -- 'No history reported'                              (CIS Credit Report Cross Reference Guide (ARF/XML/JSON), p. 49)",
    "  - TransUnion 'X' -- 'no data received / dispute / hold / unrated'      (TU4.0 User Guide, p. 840)",
]

_EQUIFAX_NOTES = _GENERAL_NOTES + [
    '',
    'Equifax specifics:',
    '  - Pattern columns used: RATE_STATUS_CODE, PAYMENT_HISTORY_1_24, PAYMENT_HISTORY_25_36, PAYMENT_HISTORY_37_48.',
    '  - Date column: DATE_REPORTED (parsed as %m%d%Y -> rptDate by MapperV2).',
    "  - Equifax NaN-fills payment-history columns with '*' upstream, so zest_payment_pattern is always padded to the trim length (48 chars).",
    '  - trimmed.str.len() == month_range for every Equifax row -- the len() part of the NEW method is a no-op here.',
    "  - The denominator difference comes entirely from subtracting '*' counts -- both trailing (pre-account-open positions) and mid-string (in-history gaps where the bureau didn't report a rating).",
]

_EXPERIAN_NOTES = _GENERAL_NOTES + [
    '',
    'Experian specifics -- why the new method requires using trimmed.str.len() instead of nominal month_range:',
    '  - Pattern column: PAYMENT_PROFILE.',
    '  - Date column: Y2K_BALANCE_DATE (parsed as %m%d%Y -> rptDate by MapperV2).',
    "  - The Experian asset sets placeholder='-' on PaymentPatternsAggregatorV2. The aggregator's _remove_placeholder step strips every '-' from the payment-pattern string BEFORE we run the by-hand count below.",
    '  - For a sparse-history account (e.g. PAYMENT_PROFILE="0--------", 3 months reported), after stripping the dashes the string is just "0" -- length 1, not 24.',
    "  - OLD: eff_old = 24 - 0 = 24 (denominator counts 23 positions that don't exist).",
    "  - NEW: eff_new = trimmed.str.len() - 0 - 0 = 1 (denominator matches observed history length).",
    "  - Without the len() change the NEW formula would compute eff_new = 24 - 0 - 0 = 24 too (because '-' has already been stripped). The len() change is essential for Experian.",
]

_TRANSUNION_NOTES = _GENERAL_NOTES + [
    '',
    'TransUnion specifics -- why the new method requires using trimmed.str.len() instead of nominal month_range:',
    '  - Pattern columns: ppt_status (derived from MANNER_OF_PAYMENT via the asset\'s status-code mapping), PAYMENT_PATTERN.',
    '  - Date column: EFFECTIVE_DATE (parsed as %Y%m%d -> rptDate by MapperV2).',
    '  - TransUnion does NOT NaN-fill PAYMENT_PATTERN. Per the asset, the NaN fallback is the empty string "".',
    '  - A consumer with a short history (e.g. account opened 4 months ago, reporting monthly) has a payment-pattern string only 4 chars long -- no padding character.',
    "  - OLD: eff_old = 24 - 0 = 24 (denominator counts 20 positions that don't exist).",
    "  - NEW: eff_new = trimmed.str.len() - 0 - 0 = 4 (denominator matches observed history length).",
    "  - TransUnion's 'X' covers both 'no data received' and dispute/hold/unrated cases. Trended subscribers may also emit 'Y' for reporting gaps -- adding 'Y' to missing_data_chars in the asset is a one-line follow-up if needed.",
]

DESCRIPTIONS = {
    'overall':    _GENERAL_NOTES,
    'equifax':    _EQUIFAX_NOTES,
    'experian':   _EXPERIAN_NOTES,
    'transunion': _TRANSUNION_NOTES,
}
DESCRIPTIONS = {k: v for k, v in DESCRIPTIONS.items() if k in TABS}
for tab in DESCRIPTIONS:
    print(f'{tab}: {len(DESCRIPTIONS[tab])} description lines')

overall: 27 description lines
equifax: 34 description lines
experian: 36 description lines
transunion: 36 description lines


In [21]:
F.write_eval_excel(TABS, OUT_XLSX, descriptions=DESCRIPTIONS)
print(f'wrote raw       -> {OUT_XLSX}')

fmt = F.format_eval_excel(OUT_XLSX, OUT_XLSX_FMT)
print(f'wrote formatted -> {fmt}')

wrote raw       -> /home/jag/payment-processor-research/payment_processing_research_data/analysis/denominator_analysis_train.xlsx
wrote formatted -> /home/jag/payment-processor-research/payment_processing_research_data/analysis/denominator_analysis_train_formatted.xlsx


In [22]:
from IPython.display import FileLink, display

print(f'file: {OUT_XLSX_FMT}  ({OUT_XLSX_FMT.stat().st_size / (1024 * 1024):.1f} MB)')
display(FileLink(str(OUT_XLSX_FMT)))
display(FileLink(str(OUT_XLSX)))

file: /home/jag/payment-processor-research/payment_processing_research_data/analysis/denominator_analysis_train_formatted.xlsx  (0.0 MB)


/home/jag/payment-processor-research/payment_processing_research_data/analysis/denominator_analysis_train_formatted.xlsx

/home/jag/payment-processor-research/payment_processing_research_data/analysis/denominator_analysis_train.xlsx